In [7]:
import torch
from torch import nn
import math
from torch.utils.tensorboard import SummaryWriter
import collections
from torch.utils.data import DataLoader, TensorDataset

utils

In [ ]:
class Vocab:
    def __init__(self, tokens,reserved_tokens=None):
        if reserved_tokens is None:
            reserved_tokens = []    
        # 去重并排序
        if tokens and isinstance(tokens[0], (list, tuple)):
            tokens = [
                token
                for sentence in tokens
                for token in sentence
            ]

        unique_tokens = sorted(set(tokens))
        # 建立索引映射
        self.idx_to_token = ["<unk>"] + reserved_tokens       
        for token in unique_tokens:
            if token not in self.idx_to_token:
                self.idx_to_token.append(token)
        self.token_to_idx = {token: idx for idx, 
                             token in enumerate(self.idx_to_token)}

    #token转idx
    def __getitem__(self, tokens):
        #如果传入的是列表或者元组
        if isinstance(tokens, (list, tuple)):
            return [self[t] for t in tokens]
        #第二个值是查询不到时的默认值
        return self.token_to_idx.get(
            tokens,
            self.token_to_idx["<unk>"]
        )

    #idx转token
    def to_tokens(self, indices):
        """把单个编号或编号列表转换为词元。"""
        if isinstance(indices, (list, tuple)):
            return [
                self.idx_to_token[index]
                for index in indices
            ]

        return self.idx_to_token[indices]

    def __len__(self):
        """返回词表大小"""
        return len(self.idx_to_token)
def truncate_pad(tokens, num_steps, padding_token):
    if len(tokens) > num_steps:
        return tokens[:num_steps]

    return tokens + [padding_token] * (num_steps - len(tokens))
def sequence_mask(X, valid_len, value=0):
    #maxlen是X的第二个维度的长度，也就是时间步的长度
    maxlen = X.size(1)
    #torch.arrage(maxlen)生成一个从0到maxlen-1的序列
    #unsqueeze(0)将这个序列的维度扩展为(1,maxlen)，
    #valid_len.unsqueeze(1)将valid_len的维度扩展为(batch_size,1)
    #[1,5]和[5,1]经广播进行比较
    mask = torch.arange(maxlen, dtype=torch.float32,
                        device=X.device).unsqueeze(0) < valid_len.unsqueeze(1)
    #~mask取反
    X[~mask] = value
    return X
def masked_softmax(X, valid_lens):
    #这里的valid_Len指的是编码器的valid_len
    """通过在最后一个轴上掩蔽元素来执行softmax操作"""
    # X:[batch_size, num_queries, num_keys]，valid_lens:1D或2D张量
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            #valid_lens的维度时[num_query*batch_size]
            #repeat_interleave是每个元素重复多次
            valid_lens = torch.repeat_interleave(valid_lens, shape[1])
        else:
            valid_lens = valid_lens.reshape(-1)
        # 最后一轴上被掩蔽的元素使用一个非常大的负值替换，从而其softmax输出为0
        #x reshape后是[batchsize*numquery,numkey]
        X = sequence_mask(X.reshape(-1, shape[-1]), valid_lens,
                              value=-1e6)
        return nn.functional.softmax(X.reshape(shape), dim=-1)
def MaskedSoftmaxCELoss(pred, label, valid_len):
    #这里valid_len指的是解码器的valid_len
    weights = torch.ones_like(label)
    weights = sequence_mask(weights, valid_len)
    #交叉熵要求类别维度位于第一维
    #返回的形状是[batch_size,num_steps]
    unweighted_loss = torch.nn.CrossEntropyLoss(reduction='none')(pred.permute(0, 2, 1), label)
    #返回的是[batch_size]，每个样本的平均损失
    weighted_loss = (
    (unweighted_loss * weights).sum(dim=1)
    / weights.sum(dim=1).clamp_min(1))
    #clamp_min(1)是为了防止除以0的情况,把小于1的值都变为1
    return weighted_loss
#点积注意力机制
class DotProductAttention(nn.Module):
    """缩放点积注意力"""
    def __init__(self, dropout, **kwargs):
        super().__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    # queries的形状：(batch_size，查询的个数，d)
    # keys的形状：(batch_size，“键－值”对的个数，d)
    # values的形状：(batch_size，“键－值”对的个数，值的维度)
    # valid_lens的形状:(batch_size，)或者(batch_size，查询的个数)
    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        # 设置transpose_b=True为了交换keys的最后两个维度
        scores = torch.bmm(queries, keys.transpose(1,2)) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, valid_lens)
        #最后返回的形状是[batch_size,num_querys,value_size]
        return torch.bmm(self.dropout(self.attention_weights), values)
def bleu(pred_seq, label_seq, k):  
    """计算BLEU"""
    pred_tokens, label_tokens = pred_seq.split(' '), label_seq.split(' ')
    len_pred, len_label = len(pred_tokens), len(label_tokens)
    score = math.exp(min(0, 1 - len_label / len_pred))
    for n in range(1, k + 1):
        num_matches, label_subs = 0, collections.defaultdict(int)
        for i in range(len_label - n + 1):
            label_subs[' '.join(label_tokens[i: i + n])] += 1
        for i in range(len_pred - n + 1):
            if label_subs[' '.join(pred_tokens[i: i + n])] > 0:
                num_matches += 1
                label_subs[' '.join(pred_tokens[i: i + n])] -= 1
        score *= math.pow(num_matches / (len_pred - n + 1), math.pow(0.5, n))
    return score

model

In [ ]:
class Seq2SeqAttentionEncoder(nn.Module):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers):
        super().__init__()
        # 嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_size)
        #num_layers：生成多少个隐藏层，这里是深层RNN
        self.rnn = nn.GRU(embed_size, num_hiddens, num_layers)

    def forward(self, X):
        # 输出'X'的形状：(batch_size,num_steps,embed_size)
        X = self.embedding(X)
        #将num_steps和batch_size维度交换位置
        X = X.permute(1, 0, 2)
        output, state = self.rnn(X)
        #output保存的是最后一个隐藏层每一个时间步的隐藏状态，在注意力机制中需要
        #state保存的是每一个隐藏层最后一个时间步的隐藏状态
        # output的形状:(num_steps,batch_size,num_hiddens)
        # state的形状:(num_layers,batch_size,num_hiddens)
        return output, state
class Seq2SeqAttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        super(Seq2SeqAttentionDecoder, self).__init__(**kwargs)
        self.attention = DotProductAttention(dropout=dropout)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(
            embed_size + num_hiddens, num_hiddens, num_layers,
            dropout=dropout)
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, enc_valid_lens, *args):
        
        # hidden_state的形状为(num_layers，batch_size，num_hiddens)
        outputs, hidden_state = enc_outputs
        return (outputs.permute(1, 0, 2), hidden_state, enc_valid_lens)
        # outputs的形状为(batch_size，num_steps，num_hiddens).

    def forward(self, X, state):
        # enc_outputs的形状为(batch_size,num_steps,num_hiddens).
        # hidden_state的形状为(num_layers,batch_size,num_hiddens)
        enc_outputs, hidden_state, enc_valid_lens = state
        # 输出X的形状为(num_steps,batch_size,embed_size)
        X = self.embedding(X).permute(1, 0, 2)
        outputs, self._attention_weights = [], []
        for x in X: 
            # query的形状为(batch_size,1,num_hiddens)
            query = torch.unsqueeze(hidden_state[-1], dim=1)
            # context的形状为(batch_size,1,num_hiddens)
            #enc_outputs形状是[batch_size,num_steps,num_hiddens]
            context = self.attention(
                query, enc_outputs, enc_outputs, enc_valid_lens)
            # 在特征维度上连结
            x = torch.cat((context, torch.unsqueeze(x, dim=1)), dim=-1)
            # 将x变形为(1,batch_size,embed_size+num_hiddens)
            out, hidden_state = self.rnn(x.permute(1, 0, 2), hidden_state)
            #out是[1,batch_size,num_hiddens]
            outputs.append(out)
            self._attention_weights.append(self.attention.attention_weights)
        # 全连接层变换后，outputs的形状为
        # (num_steps,batch_size,vocab_size)
        outputs = self.dense(torch.cat(outputs, dim=0))
        #这里的output是decoder层的输出
        return outputs.permute(1, 0, 2), [enc_outputs, hidden_state,
                                          enc_valid_lens]
        #返回outputs是decoder的输出[batch_size,num_steps,vocab_size]
        #返回enc_outputs是encoder的输出[batch_size,num_step,num_hiddens]
        #enc_valid_lens是输入的有效序列
class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X,enc_valid_len):
        enc_outputs = self.encoder(enc_X)
        dec_state = self.decoder.init_state(enc_outputs,enc_valid_len)
        return self.decoder(dec_X, dec_state)

dataloader

In [ ]:
#读取语料
def read_data_nmt(file_path):
    source = []
    target = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")

            # fra.txt 某些行可能包含额外的来源信息，
            # 这里只取前两列：英文和法文
            if len(parts) < 2:
                continue

            source.append(parts[0].lower())
            target.append(parts[1].lower())

    return source, target
#将特殊空格替换为普通空格，并在标点符号前加空格
def preprocess_nmt(text):
    #替换特殊空格
    text = text.replace("\u202f", " ")
    text = text.replace("\xa0", " ")

    #在符号前面加空格
    for punctuation in [",", ".", "!", "?"]:
        text = text.replace(
            punctuation,
            " " + punctuation
        )

    return text.strip()
#将句子拆分为词元
def tokenize_nmt(sentences):
    return [
        preprocess_nmt(sentence).split()
        for sentence in sentences
    ]
#将词元转成编号，并在末尾加上eos，截断或填pad，记录有效长度
def build_array_nmt(
    lines,
    vocab,
    num_steps
):
    # 每句话先转成编号，并在末尾加入 <eos>
    lines = [
        vocab[line[:num_steps-1]] + [vocab["<eos>"]]
        for line in lines
    ]

    # 截断或填充到 num_steps
    array = torch.tensor(
        [
            truncate_pad(
                line,
                num_steps,
                vocab["<pad>"]
            )
            for line in lines
        ],
        dtype=torch.long
    )

    # 统计每句话中非 <pad> 词元的数量
    valid_len = (
        array != vocab["<pad>"]
    ).sum(dim=1)
    #这里得到的vali_len向量维度为[lines]一维的
    return array, valid_len
#创建迭代器，返回迭代器，源语言和带翻译语言词表
def load_data_nmt(
    file_path,
    batch_size,
    num_steps,
):
    # 读取英文句子和法文句子
    source_sentences, target_sentences = read_data_nmt(
        file_path
    )

    # 分词，结果是二维列表
    source_tokens = tokenize_nmt(
        source_sentences
    )
    target_tokens = tokenize_nmt(
        target_sentences
    )

    # 分别建立源语言词表和目标语言词表
    reserved_tokens = [
        "<pad>",
        "<bos>",
        "<eos>"
    ]

    src_vocab = Vocab(
        source_tokens,
        reserved_tokens=reserved_tokens
    )

    tgt_vocab = Vocab(
        target_tokens,
        reserved_tokens=reserved_tokens
    )

    # 转换为定长张量
    src_array, src_valid_len = build_array_nmt(
        source_tokens,
        src_vocab,
        num_steps
    )

    tgt_array, tgt_valid_len = build_array_nmt(
        target_tokens,
        tgt_vocab,
        num_steps
    )

    # 一个样本由四个张量组成
    dataset = TensorDataset(
        src_array,
        src_valid_len,
        tgt_array,
        tgt_valid_len
    )

    data_iter = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    return data_iter, src_vocab, tgt_vocab

train

In [ ]:
#训练函数
def train_seq2seq(net, data_iter, lr, num_epochs, tgt_vocab, device):
    #初始化参数
    def xavier_init_weights(m):
        if type(m) == nn.Linear:
            nn.init.xavier_uniform_(m.weight)
        if type(m) == nn.GRU:
            for param in m._flat_weights_names:
                if "weight" in param:
                    nn.init.xavier_uniform_(m._parameters[param])

    net.apply(xavier_init_weights)
    net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss = MaskedSoftmaxCELoss
    net.train()

    writer = SummaryWriter(log_dir='./logs')
    for epoch in range(num_epochs):
        total_loss = 0.0
        total_samples = 0
        print(f'============epoch {epoch + 1}============')
        for batch in data_iter:
            optimizer.zero_grad()
            #这里valid_len的维度为[batchsize]
            X, X_valid_len, Y, Y_valid_len = [x.to(device) for x in batch]
            #将bos改为[batch_size,1]
            bos = torch.tensor([tgt_vocab['<bos>']] * Y.shape[0],
                          device=device).reshape(-1, 1)
            #将bos和Y的前n-1个时间步拼接在一起作为解码器的输入
            dec_input = torch.cat([bos, Y[:, :-1]], 1)  # 强制教学
            #_保存的是解码器所有层的最后一个时间步的隐藏状态
            Y_hat, _ = net(X, dec_input,X_valid_len)
            l = loss(Y_hat, Y, Y_valid_len).mean()
            l.backward()      # 损失函数的标量进行“反向传播”
            #进行梯度剪裁
            torch.nn.utils.clip_grad_norm_(
                net.parameters(),
                max_norm=1.0
            )
            optimizer.step()
            total_loss += l.item() * Y.shape[0]
            total_samples += Y.shape[0]
        avg_loss = total_loss / total_samples
        writer.add_scalar('Loss/Train', avg_loss, epoch+1)
    writer.close()
def predict_seq2seq(net, src_sentence, src_vocab, tgt_vocab, num_steps,
                    device):
    '''
    src_sentence: str, 输入的源语言句子
    src_vocab: Vocab, 源语言词汇表
    tgt_vocab: Vocab, 目标语言词汇表
    num_steps: int, 输出序列的最大长度
    device: torch.device, 计算设备
    '''
    # 在预测时将net设置为评估模式
    net.eval()
    #将原句子小写按单词切分加上eos的编码
    #src_tokens的形状是[num_steps]
    src_tokens = src_vocab[
    preprocess_nmt(
        src_sentence.lower()
    ).split()] + [src_vocab["<eos>"]]
    #获取encoder输入的有效长度
    enc_X_valid_len = torch.tensor(
    [min(len(src_tokens), num_steps)],
    dtype=torch.long,device=device)
    #处理序列阶段，如果太长截断，太短加padding tokens
    src_tokens = truncate_pad(src_tokens, num_steps, src_vocab['<pad>'])
    
    with torch.no_grad():
        #添加batch_size维度，符合encoder输入维度
        enc_X = torch.unsqueeze(
            torch.tensor(src_tokens, dtype=torch.long, device=device), dim=0)
        enc_outputs = net.encoder(enc_X)
        #获得解码器的初始状态
        dec_state = net.decoder.init_state(enc_outputs,enc_X_valid_len)
        #为输入的bos添加batch_size维度，符合decoder输入维度
        dec_X = torch.unsqueeze(torch.tensor(
            [tgt_vocab['<bos>']], dtype=torch.long, device=device), dim=0)
        output_seq = []
        for _ in range(num_steps):
            Y, dec_state = net.decoder(dec_X, dec_state)
            # 我们使用具有预测最高可能性的词元，作为解码器在下一时间步的输入
            dec_X = Y.argmax(dim=2)
            #获得预测的整数
            pred = dec_X.squeeze(dim=0).type(torch.int32).item()
            # 一旦序列结束词元被预测，输出序列的生成就完成了
            if pred == tgt_vocab['<eos>']:
                break
            output_seq.append(pred)
        return ' '.join(tgt_vocab.to_tokens(output_seq))

main

In [12]:
def main():   
    file_path = "./data/fra-eng/fra.txt"

    batch_size = 64
    num_steps = 10

    data_iter, src_vocab, tgt_vocab = load_data_nmt(
        file_path=file_path,
        batch_size=batch_size,
        num_steps=num_steps,
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available()
        else "cpu"
    )

    embed_size = 32
    num_hiddens = 32
    num_layers = 2

    encoder = Seq2SeqAttentionEncoder(
        vocab_size=len(src_vocab),
        embed_size=embed_size,
        num_hiddens=num_hiddens,
        num_layers=num_layers
    )

    decoder = Seq2SeqAttentionDecoder(
        vocab_size=len(tgt_vocab),
        embed_size=embed_size,
        num_hiddens=num_hiddens,
        num_layers=num_layers
    )

    net = EncoderDecoder(
        encoder,
        decoder
    )

    train_seq2seq(
        net=net,
        data_iter=data_iter,
        lr=0.005,
        num_epochs=50,
        tgt_vocab=tgt_vocab,
        device=device
    )
    return net, src_vocab, tgt_vocab, num_steps, device
if __name__ == '__main__':
    # 训练完成后进行翻译
    test_sentences = ['go .', 
            "i lost .", 
            'he\'s calm .', 
            'i\'m home .']
    tgt_sentence = ['va !', 
                    'j\'ai perdu .', 
                    'il est calme .', 
                    'je suis chez moi .']
    net, src_vocab, tgt_vocab, num_steps, device = main()

    for src_sentence, target_sentence in zip(
        test_sentences,tgt_sentence):
        translation = predict_seq2seq(
            net=net,
            src_sentence=src_sentence,
            src_vocab=src_vocab,
            tgt_vocab=tgt_vocab,
            num_steps=num_steps,
            device=device
        )
        print('BLEU',bleu(translation,tgt_sentence,2))
        print(
            f"{src_sentence} -> {translation}"
        )

============epoch 1============


KeyboardInterrupt: 